# CS-781 Capstone — Phase 1: Decoder-Only Transformer with GQA and a KV Cache

This notebook trains and submits ONE architecture end-to-end on Kaggle:
1. Get the legal text data
2. Turn text into numbers (a BPE subword tokenizer, trained on this corpus)
3. Build the Decoder-Only Transformer with GQA and a KV Cache
4. Train it (checkpoints saved to `/kaggle/working`)
5. Generate the 128-word continuations for the test set
6. Package `submission.zip` to upload to Codabench

**Workflow:** run this notebook with `QUICK_TEST = True` first (Settings
cell) to confirm everything works on a small slice of data in a few
minutes. Then set `QUICK_TEST = False` and run it again for real. This
is one of 4 separate notebooks (decoder-only Transformer, encoder-decoder
Transformer, LSTM, CNN) — the brief requires comparing 4 substantially
different architectures, so run all 4 the same way and compare their
local evaluation scores (Step 15) before deciding what to submit.

On top of that, this notebook also exposes `NORM_TYPE`, `NORM_POSITION` and `OPTIMIZER_TYPE` in the Settings cell, so you can systematically try
different normalization / attention / optimizer choices (per the lecture
slides on comparing attentions and normalizations) and compare them the
same way you compare architectures.

**Before running, in the panel on the right:**
- **Accelerator** → pick a GPU (e.g. "GPU T4 x2").
- **Internet** → turn ON (needed to download the dataset from Hugging Face
  and to `pip install` a couple of packages).

---
**THIS VARIANT: 160M words, ONE epoch.** Same model, A1 BPE tokenizer, GQA and RoPE as the 80M-word/2-epoch notebook, but the corpus is
**160M words (40M per source)** and it trains for **1 epoch** (every document is seen once, so the total number of training steps is about the same).
Everything that goes to the Hub carries `RUN_TAG = "160Mw_1ep"` in its name, so nothing collides with the 80M/2-epoch run:
checkpoint `..._160Mw_a1bpe_160Mw_1ep_checkpoint.pt`, tokenizer `a1bpe_tokenizer_160Mw_1ep_...json`, run log `run_log_gqa_kvcache_160Mw_1ep_...jsonl`.
(The 160M-word tokenizer is a different tokenizer from the 80M one, because it is trained on a different corpus.)

## 1. Settings

In [1]:
# ----------------------------------------------------------------
# QUICK_TEST lets you sanity-check the whole notebook in a few
# minutes before committing GPU time to a real run.
#
#   1. First run: leave QUICK_TEST = True. Run every cell top to
#      bottom. This trains on a small slice of data for 1 epoch,
#      just to prove nothing crashes and a submission.zip comes out
#      the other end.
#   2. Once that works: set QUICK_TEST = False and re-run every cell.
#      This is your real run, on much more data, for more epochs.
# ----------------------------------------------------------------
QUICK_TEST = False

if QUICK_TEST:
    MAX_WORDS_FOR_TRAINING = 200_000   # a small slice, just to test the pipeline
    NUM_EPOCHS = 1
else:
    MAX_WORDS_FOR_TRAINING = 160000000   # increase this further if your GPU/time budget allows
    NUM_EPOCHS = 1       # 160M words x 1 epoch (the other notebook: 80M words x 2 epochs)

VOCAB_SIZE = 40000        # BPE subword vocabulary size -- larger than a word-level vocab needs to be, since subwords cover rare words without <unk>
CONTEXT_LEN = 2048       # tokens the model sees at once (a 512-word prompt is roughly 700-1000 BPE tokens)
BATCH_SIZE = 2           # examples per step (long 2048-token windows: keep this small, or the GPU runs out of memory)
LEARNING_RATE = 3e-4
TEMPERATURE = 0.3        # 0 = greedy (argmax). Small values (0.1-0.5) = nearly greedy with a little
                         # randomness; see the decoding notes in the generation section and the sweep in Step 15

# ----------------------------------------------------------------
# Research knobs -- this is the whole point of a research project:
# change ONE of these, re-run, and compare. See lecture slides on
# "Comparing Attentions" and normalization for background.
# ----------------------------------------------------------------
NORM_TYPE = "layernorm"  # "layernorm" | "rmsnorm" | "deepnorm"
# layernorm = the standard normalization used in most Transformers
# rmsnorm   = a cheaper norm that skips re-centering the mean (used in
#             LLaMA and others)
# deepnorm  = a post-norm recipe designed to make very deep stacks stable
#             (scales the residual connection by a fixed alpha and uses a
#             special weight init). deepnorm always places the norm AFTER
#             the sub-layer (it ignores NORM_POSITION below).

NORM_POSITION = "pre"    # "pre" | "post"
# pre  = normalize BEFORE each sub-layer (norm(x) -> sublayer -> add residual).
#        This is what most modern Transformers (GPT-2 and later) use --
#        it trains more stably, especially for deep models.
# post = normalize AFTER adding the residual (sublayer -> add -> norm(...)).
#        This is what the original 2017 Transformer paper used.

OPTIMIZER_TYPE = "adamw"  # "adamw" | "muon"
# adamw = the standard optimizer almost everyone uses.
# muon  = a newer optimizer that "orthogonalizes" the update for every 2D+
#         weight matrix (biases/embeddings/norm weights still use adamw
#         underneath -- that's the standard muon recipe).

# Checkpoint filename includes QUICK_TEST, NORM_TYPE/POSITION
# and OPTIMIZER_TYPE so different experiment settings never overwrite each
# other's checkpoints (this also means switching ATTENTION_TYPE between runs
# always starts a fresh checkpoint instead of colliding with a differently-
# shaped one from another attention type).
RUN_TAG = "160Mw_1ep"   # goes into every Hub file name (checkpoint, tokenizer, run log) so this run never collides with the 80M/2-epoch run
mode_tag = "quicktest" if QUICK_TEST else "full"
CHECKPOINT_PATH = (
    f"/kaggle/working/phase1_decoder_only_gqa_kvcache_{mode_tag}_{NORM_TYPE}_{NORM_POSITION}"
    f"_{OPTIMIZER_TYPE}_checkpoint.pt"
)

## 2. Make sure the working folder exists (and a note on Kaggle persistence)

Checkpoints go to `/kaggle/working`. That folder survives you re-running
cells, and even a "Restart session", within the same notebook edit session.

If you come back on a **completely new day / new session** and want to
resume: click **Save Version** before you stop working. Next time, use
**Add Input → Notebook Output Files → (this notebook's last saved version)**
to pull the checkpoint back in, then copy it into `/kaggle/working` before
re-running the training cell.

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. One-time setup, then install and import what we need

**The dataset is GATED** -- before this notebook can download anything, do
this once per Kaggle **account** (each of your two accounts needs its own):

1. While logged into Hugging Face, open
   https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone and
   accept the dataset's access conditions (there's a form to agree to on
   that page -- without this, downloads below get an auth error).
2. Create a **read** access token at https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name
   it exactly `HF`, and paste the token as its value.

Do this before running the cell below -- it logs in using that secret.

In [3]:
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Sync checkpoints with a Hugging Face model repo

Instead of manually downloading/uploading `.pt` files between Kaggle
sessions or accounts, checkpoints are pushed to a Hugging Face **model**
repo after every epoch, and pulled back down automatically if this session
doesn't already have a local one. The filename already encodes the full
config (architecture, QUICK_TEST mode, norm/attention/optimizer settings --
see the Settings cell), so the repo doubles as a self-documenting archive
of every run you've tried -- handy when writing up the report.

**One-time setup (separate from the read-only `HF` secret used for the
gated dataset):**
1. Create a Hugging Face **model** repo (e.g. at
   huggingface.co/new -> "Model"), private or public, your choice.
2. Create a **write**-scoped access token at
   https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons -> Secrets -> Add a new secret**,
   name it exactly `HF_WRITE`, and paste that token as its value. (Keep
   the existing `HF` secret as-is -- that one stays read-only, just for
   downloading the dataset.)

In [4]:
from huggingface_hub import HfApi, create_repo
import shutil

HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)
create_repo(HF_CHECKPOINT_REPO, token=HF_WRITE_TOKEN, repo_type="model", exist_ok=True)


def push_checkpoint_to_hub(local_path, quiet=False):
    """Upload the file at local_path (a checkpoint, the tokenizer, or the run log)
    to the HF repo, under its own filename (which already encodes the full config
    -- see Settings). Each upload overwrites the previous version of that file."""
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=os.path.basename(local_path),
        repo_id=HF_CHECKPOINT_REPO,
        repo_type="model",
        token=HF_WRITE_TOKEN,
    )
    if not quiet:
        print(f"Pushed {os.path.basename(local_path)} to https://huggingface.co/{HF_CHECKPOINT_REPO}"
              f"/blob/main/{os.path.basename(local_path)}")


def pull_from_hub(local_path):
    """Quietly fetch local_path's file (by basename) from the HF repo if it is not
    on disk yet. Returns True if the file exists locally afterwards."""
    if os.path.exists(local_path):
        return True
    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO, filename=os.path.basename(local_path),
            repo_type="model", token=HF_WRITE_TOKEN,
        )
        shutil.copy(downloaded_path, local_path)
        print(f"Pulled {os.path.basename(local_path)} from the Hub")
        return True
    except Exception:
        return False


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (e.g. a fresh Kaggle session or a
    different account), try to fetch a matching one from the HF repo
    before falling back to starting from scratch."""
    if os.path.exists(local_path):
        return
    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO, filename=os.path.basename(local_path),
            repo_type="model", token=HF_WRITE_TOKEN,
        )
        shutil.copy(downloaded_path, local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Checkpoint sync configured -> https://huggingface.co/{HF_CHECKPOINT_REPO}")

# ---- Run log: a running record of what this project did, for the final report ----
# Every important step appends one JSON line (time, what happened, the numbers) to
# RUN_LOG_PATH. The file is pushed to the Hub together with the checkpoints and pulled
# back at the start of a new session, so a run that spans several sessions (or both
# Kaggle accounts) keeps ONE continuous log. Read it with: print(open(RUN_LOG_PATH).read())
import time as _time, datetime as _dt
RUN_LOG_PATH = f"/kaggle/working/run_log_gqa_kvcache_{RUN_TAG}_{mode_tag}.jsonl"
pull_from_hub(RUN_LOG_PATH)


def log_event(kind, **fields):
    """Append one event to the run log (never raises: logging must not stop a training run)."""
    try:
        record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
    except Exception as e:
        print(f"(run log write failed: {type(e).__name__})")


def sync_run_log():
    try:
        push_checkpoint_to_hub(RUN_LOG_PATH, quiet=True)
    except Exception as e:
        print(f"(run log push failed: {type(e).__name__})")


log_event("session_start", quick_test=QUICK_TEST, vocab_size=VOCAB_SIZE, context_len=CONTEXT_LEN,
          batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
          max_words=MAX_WORDS_FOR_TRAINING, norm=f"{NORM_TYPE}/{NORM_POSITION}", optimizer=OPTIMIZER_TYPE,
          device=device, gpu=(torch.cuda.get_device_name(0) if device == "cuda" else None))
print(f"Run log: {RUN_LOG_PATH}")

Checkpoint sync configured -> https://huggingface.co/kavincg23/cs781-checkpoints
Run log: /kaggle/working/run_log_gqa_kvcache_160Mw_1ep_full.jsonl


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 4. Download the data files from Hugging Face

This dataset has no `datasets.load_dataset(...)` config -- it's raw JSON
files, one per source, downloaded directly (the same way the official
starter notebook does it). Make sure **Internet** is turned ON in the
settings panel on the right.

**Important:** the 4 train files together hold the *entire* >4.5B-token
corpus. `json.load()`-ing even one of them fully into memory is enough to
exceed Kaggle's RAM and crash the notebook ("tried to allocate more memory
than is available") -- and that happens regardless of `MAX_WORDS_FOR_TRAINING`,
since that setting only limits how many words we *use*, not how much we'd
have loaded first. So instead of `json.load`, we use `ijson` to stream
through each file's JSON array one document at a time, only ever holding a
handful of documents in memory. The print statement below just peeks at
the first document, to let you SEE the real shape of the data.

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

first_train_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILES[0],
                                    repo_type="dataset", token=HF_TOKEN)
with open(first_train_path, "r", encoding="utf-8") as f:
    preview_record = next(ijson.items(f, "item"))
print("Example document (from the first train file):")
print(preview_record)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.4 MB/s eta 0:00:00


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Example document (from the first train file):
{'title': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996', 'category': 'Greater Bengaluru City Corporation - Act', 'year': 1972, 'url': 'https://indiankanoon.org/doc/179573747/', 'text': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission granted by the Corporation contrary to the Zonal Regulations then in force was questioned in the High Court and the Division Bench of the High Court held that the permission granted in violation of the Zonal Regulation was illegal.

### Check the field name above

Look at the example printed above -- it should be a dict like
`{'text': '...'}` (confirmed from the official starter notebook). Set
`TEXT_FIELD` below to whatever key actually holds the legal text --
change it if the printed example above used a different name.

In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

### Download and load the test prompts (the ones we must generate continuations for)

`test.json` is a single JSON object keyed by ID (`{"1": {"text": "..."}, ...}`),
not a list -- that's already the exact shape Codabench wants predictions in.

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json: 0.00B [00:00, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


### Stream through the training files, collecting words until we have enough

This walks through `TRAIN_FILES` one document at a time (still via `ijson`,
never loading a full file into memory) and stops as soon as it has collected
`MAX_WORDS_FOR_TRAINING` words -- so a `QUICK_TEST` run only ever reads a
small slice of the corpus off disk, not all >4.5B tokens of it.

In [8]:
import array, random, time
SAMPLING_SEED = 42      # fixed seed: the same documents are chosen on every run
# Whole documents held out for EVALUATION (words per source; never trained on). The Supreme Court set is
# the largest because the test prompts look like Supreme Court text.
HELDOUT_WORDS = {'acts': 500000, 'high_courts': 500000, 'supreme_court': 2000000, 'tribunals': 500000}
HELDOUT_MIN_DOC_WORDS = 640       # long enough for a 512-word prompt + 128-word continuation, like the test examples
HELDOUT_MAX_WORDS_PER_DOC = 6000  # cap so a few enormous documents do not dominate
HELDOUT_FILES = {}                # source name -> file of held-out documents (one per line); filled below
print("Streaming through the training files, collecting words until we have enough...")

# Words are written straight to a FILE as we stream through the sources,
# instead of being collected into one giant Python list/string first. The
# BPE tokenizer below trains directly from this file (it streams it from
# disk in blocks) rather than us handing it one huge in-memory
# string -- this is what lets tokenizer training handle a large word count
# without running out of RAM (building the string in Python first was the
# actual bottleneck, not the tokenizer trainer itself).
CORPUS_FILE = "/kaggle/working/bpe_training_corpus.txt"

if QUICK_TEST:
    # fast pipeline check -- just fill up to MAX_WORDS_FOR_TRAINING from
    # whichever source(s) come first, same as before, just writing to the
    # corpus file instead of a Python list.
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            if word_count >= MAX_WORDS_FOR_TRAINING:
                break
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_words = record[TEXT_FIELD].split()
                    remaining = MAX_WORDS_FOR_TRAINING - word_count
                    if remaining <= 0:
                        break
                    if len(doc_words) > remaining:
                        doc_words = doc_words[:remaining]
                    out_f.write(" ".join(doc_words) + " ")
                    word_count += len(doc_words)
                    if word_count >= MAX_WORDS_FOR_TRAINING:
                        break
            print(f"  ...after {filename}: {word_count} words collected so far")
else:
    # Full run: pull a FIXED QUOTA from EACH of the 4 sources, instead of
    # filling up sequentially (which would silently train on only the
    # first source(s)). This is the "data mixing across sources"
    # experiment the brief expects -- change the quotas below to try a
    # different mix, and document what you tried.
    # Within each source the documents are a SEEDED RANDOM sample of the whole file
    # (not the first documents in it), so a file that happens to be sorted by court,
    # year, etc. cannot bias what the model sees. QUICK_TEST still takes the first documents.
    SOURCE_WORD_QUOTAS = {
    "Phase_1_Pretraining/Train/train_acts.json": 40000000,
    "Phase_1_Pretraining/Train/train_high_courts.json": 40000000,
    "Phase_1_Pretraining/Train/train_supreme_court.json": 40000000,
    "Phase_1_Pretraining/Train/train_tribunals.json": 40000000,
    }
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            quota = SOURCE_WORD_QUOTAS.get(filename, 0)
            if quota <= 0:
                continue
            short = os.path.basename(filename)[len("train_"):-len(".json")]
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)

            # PASS 1: count the words of every document (the text itself is thrown
            # away immediately, so memory stays tiny even for the multi-GB files).
            t0 = time.time()
            doc_lengths = array.array("l")
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_lengths.append(len(record[TEXT_FIELD].split()))
            print(f"  {filename}: {len(doc_lengths):,} documents, {sum(doc_lengths):,} words "
                  f"(counted in {time.time() - t0:.0f}s)")

            # A seeded random order over ALL documents of the file: every document -- and
            # so every word -- is equally likely to be picked wherever it sits in the file,
            # and the same seed always picks the same documents.
            order = list(range(len(doc_lengths)))
            random.Random(SAMPLING_SEED).shuffle(order)

            # 1) HELD-OUT documents first (never trained on, never used for the tokenizer):
            #    whole documents long enough to hold a 512-word prompt + 128-word continuation,
            #    like the test examples. Used only for evaluation (Step 15).
            held_target = HELDOUT_WORDS.get(short, 0)
            heldout_ids, held_words = set(), 0
            for i in order:
                if held_words >= held_target:
                    break
                if doc_lengths[i] >= HELDOUT_MIN_DOC_WORDS:
                    heldout_ids.add(i)
                    held_words += min(doc_lengths[i], HELDOUT_MAX_WORDS_PER_DOC)

            # 2) TRAINING documents: continue down the same random order (skipping the
            #    held-out ones) until the word quota is met; the last one is truncated.
            chosen, planned = {}, 0
            for i in order:
                if planned >= quota:
                    break
                if i in heldout_ids:
                    continue
                keep = min(doc_lengths[i], quota - planned)
                if keep > 0:
                    chosen[i] = keep
                    planned += keep
            del order

            # PASS 2: write the chosen documents (in file order) to the corpus file, and the
            # held-out ones (one document per line) to their own file.
            source_word_count = 0
            remaining_docs, remaining_held = len(chosen), len(heldout_ids)
            held_f = None
            if heldout_ids:
                heldout_path = os.path.join(os.path.dirname(CORPUS_FILE), f"heldout_{short}.txt")
                held_f = open(heldout_path, "w", encoding="utf-8")
                HELDOUT_FILES[short] = heldout_path
            with open(local_path, "r", encoding="utf-8") as f:
                for i, record in enumerate(ijson.items(f, "item")):
                    keep = chosen.get(i)
                    if keep is None and i not in heldout_ids:
                        continue
                    doc_words = record[TEXT_FIELD].split()
                    if keep is not None:
                        out_f.write(" ".join(doc_words[:keep]) + " ")
                        source_word_count += min(keep, len(doc_words))
                        remaining_docs -= 1
                    else:
                        held_f.write(" ".join(doc_words[:HELDOUT_MAX_WORDS_PER_DOC]) + "\n")
                        remaining_held -= 1
                    if remaining_docs == 0 and remaining_held == 0:
                        break
            if held_f is not None:
                held_f.close()
            word_count += source_word_count
            print(f"  ...collected {source_word_count:,} words from {len(chosen):,} randomly chosen documents of "
                  f"{filename} (target {quota:,}); held out {len(heldout_ids):,} whole documents for evaluation")
    MAX_WORDS_FOR_TRAINING = word_count  # actual total = sum of quotas above

print(f"\nUsing {word_count} words for training (written to {CORPUS_FILE})")

Streaming through the training files, collecting words until we have enough...
  Phase_1_Pretraining/Train/train_acts.json: 18,826 documents, 119,638,348 words (counted in 10s)
  ...collected 40,000,000 words from 6,615 randomly chosen documents of Phase_1_Pretraining/Train/train_acts.json (target 40,000,000); held out 140 whole documents for evaluation


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

  Phase_1_Pretraining/Train/train_high_courts.json: 2,536,738 documents, 1,594,919,242 words (counted in 127s)
  ...collected 40,000,000 words from 63,827 randomly chosen documents of Phase_1_Pretraining/Train/train_high_courts.json (target 40,000,000); held out 249 whole documents for evaluation


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

  Phase_1_Pretraining/Train/train_supreme_court.json: 55,549 documents, 210,205,151 words (counted in 16s)
  ...collected 40,000,000 words from 10,189 randomly chosen documents of Phase_1_Pretraining/Train/train_supreme_court.json (target 40,000,000); held out 609 whole documents for evaluation


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

  Phase_1_Pretraining/Train/train_tribunals.json: 869,132 documents, 975,917,446 words (counted in 86s)
  ...collected 40,000,000 words from 35,848 randomly chosen documents of Phase_1_Pretraining/Train/train_tribunals.json (target 40,000,000); held out 273 whole documents for evaluation

Using 160000000 words for training (written to /kaggle/working/bpe_training_corpus.txt)


## 5. Train a BPE (subword) tokenizer on the legal text -- our own implementation, from scratch

A word-level tokenizer with a small vocabulary throws away rare words as `<unk>`, which is bad for legal
text (case names, section numbers, Latin terms). **BPE (Byte-Pair Encoding)** instead learns to split rare
words into reusable pieces, so almost nothing becomes `<unk>`.

The tokenizer below is **not imported from any library**: it is the pure-Python byte-level BPE from
Assignment 1 (`BPETokenizer`), trained directly on this notebook's own corpus:

* words are split on whitespace and each word starts as its UTF-8 **bytes** plus an end-of-word marker
  `</w>`, so merges never cross word boundaries (and any text can be spelled out);
* training repeatedly merges the most frequent adjacent pair of symbols (ties: lexicographically smallest
  pair), until the vocabulary reaches `VOCAB_SIZE` or no pair occurs twice;
* encoding applies the learned merges to each word in the order they were learned; `decode` re-joins the
  words with single spaces. Special tokens: `<pad>`=0, `<unk>`=1, `<s>`=2, `</s>`=3.

**What is different from the Assignment-1 file, and why.** Only the *speed and memory* machinery, not the
algorithm: (1) word frequencies are counted by streaming the corpus file in blocks instead of holding the
text in memory; (2) the next merge is picked with a max-heap instead of scanning every pair each time
(scanning ~10^5-10^6 pairs for each of ~40,000 merges would take days on an 160M-word corpus); (3) each
word's token ids are cached, so the 160M-word corpus is encoded with one dictionary lookup per word. The
heap version selects exactly the same merge at every step as the Assignment-1 scan (same tie-break rule);
this was checked against the original implementation on random corpora, comparing the merge lists,
vocabulary, encodings and decodings.

Because it is plain deterministic Python, the same corpus and `VOCAB_SIZE` always give the same
vocabulary, which is what lets a saved checkpoint be resumed later. Training this in pure Python on 160M
words takes a while (progress is printed; not yet timed on the real corpus). `BPE_MIN_WORD_FREQ = 1` uses
every word type exactly like Assignment 1; raising it to 2 skips words that occur only once when *learning*
merges (faster and lighter, slightly different merges); encoding always handles every word.

**Saving the tokenizer.** Training this tokenizer is the slowest pre-training step, so the trained merges are
saved (`a1bpe_tokenizer_<mode>_v<vocab>_<words>w_seed<seed>_minf<n>.json`, in `/kaggle/working`) and pushed to the
same Hugging Face repo as the checkpoints. The file name records everything the tokenizer depends on (corpus size,
sampling seed, vocabulary size, `BPE_MIN_WORD_FREQ`). On a re-run -- or from the other Kaggle account -- the cell finds
the file and loads it instead of training; change any of those settings and it trains a new one.

In [9]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)


# The tokenizer depends only on the training corpus, VOCAB_SIZE and BPE_MIN_WORD_FREQ, so its file name records
# exactly those (the corpus is fixed by the sampling seed, the quotas and the word count). If a file with this name is
# on disk or on the Hub it is loaded (skipping the slow pure-Python training); otherwise it is trained, saved and pushed.
TOKENIZER_PATH = (f"/kaggle/working/a1bpe_tokenizer_{RUN_TAG}_{mode_tag}_v{VOCAB_SIZE}_{word_count}w"
                  f"_seed{SAMPLING_SEED}_minf{BPE_MIN_WORD_FREQ}.json")
t_start = time.time()
if pull_from_hub(TOKENIZER_PATH):
    print(f"Loading the saved tokenizer {os.path.basename(TOKENIZER_PATH)} (skipping training)...")
    bpe_tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    tokenizer_source = "loaded"
else:
    print("Training the BPE tokenizer (our own implementation, pure Python)...")
    bpe_tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE).fit_file(CORPUS_FILE, min_word_freq=BPE_MIN_WORD_FREQ)
    bpe_tokenizer.save(TOKENIZER_PATH, meta={"word_count": word_count, "sampling_seed": SAMPLING_SEED,
                                             "min_word_freq": BPE_MIN_WORD_FREQ, "requested_vocab": VOCAB_SIZE})
    try:
        push_checkpoint_to_hub(TOKENIZER_PATH)
    except Exception as e:
        print(f"(could not push the tokenizer to the Hub: {type(e).__name__} -- it is still saved locally)")
    tokenizer_source = "trained"
print(f"Tokenizer {tokenizer_source} in {time.time() - t_start:.0f}s")
log_event("tokenizer", source=tokenizer_source, seconds=round(time.time() - t_start), vocab_size=bpe_tokenizer.get_vocab_size(),
          merges=len(bpe_tokenizer.merges), words=word_count, file=os.path.basename(TOKENIZER_PATH))


def encode(text):
    """Turn a string into a list of BPE token IDs."""
    return bpe_tokenizer.encode(text)


def decode(ids):
    """Turn a list of BPE token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())
if bpe_tokenizer.get_vocab_size() < VOCAB_SIZE:
    print(f"NOTE: fewer than VOCAB_SIZE={VOCAB_SIZE} merges were possible; the model uses the real size above.")

# Encode the whole training corpus (streamed from disk, one id-cache lookup per word). The array is named
# all_word_ids for consistency with the other notebooks; it holds BPE-token ids, not one id per word.
t_start = time.time()
all_word_ids = bpe_tokenizer.encode_file(CORPUS_FILE)
print(f"{word_count} words became {len(all_word_ids)} BPE tokens "
      f"({len(all_word_ids) / word_count:.2f} tokens per word) in {time.time() - t_start:.0f}s")
log_event("encoded_corpus", words=word_count, tokens=len(all_word_ids), tokens_per_word=round(len(all_word_ids) / word_count, 3),
          seconds=round(time.time() - t_start))

Training the BPE tokenizer (our own implementation, pure Python)...
  counted 160,000,000 words (2,468,793 distinct) in 46s
  base byte alphabet: 205 symbols; learning merges from 2,468,793 word types
  merges 1,000 | vocab 1,209 | 214s
  merges 2,000 | vocab 2,209 | 249s
  merges 3,000 | vocab 3,209 | 277s
  merges 4,000 | vocab 4,209 | 296s
  merges 5,000 | vocab 5,209 | 306s
  merges 6,000 | vocab 6,209 | 320s
  merges 7,000 | vocab 7,209 | 332s
  merges 8,000 | vocab 8,209 | 341s
  merges 9,000 | vocab 9,209 | 349s
  merges 10,000 | vocab 10,209 | 363s
  merges 11,000 | vocab 11,209 | 373s
  merges 12,000 | vocab 12,209 | 383s
  merges 13,000 | vocab 13,209 | 398s
  merges 14,000 | vocab 14,209 | 407s
  merges 15,000 | vocab 15,209 | 418s
  merges 16,000 | vocab 16,209 | 425s
  merges 17,000 | vocab 17,209 | 431s
  merges 18,000 | vocab 18,209 | 440s
  merges 19,000 | vocab 19,209 | 443s
  merges 20,000 | vocab 20,209 | 447s
  merges 21,000 | vocab 21,209 | 451s
  merges 22,000 | v

/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


160000000 words became 183754833 BPE tokens (1.15 tokens per word) in 155s


## 6. Chop the training text into fixed-size chunks

We cut the long stream of word-IDs into pieces of length `CONTEXT_LEN`
(640 words). Each piece is one training example. A model learns to predict
the next word at every position in the piece.

*Simplification:* this glues all documents together into one long stream
before cutting, so a few chunks will mix the end of one case with the start
of another. That's fine for a first version — separating documents cleanly
is one of the "task-specific preprocessing" experiments you could try later.

In [10]:
# Build the chunk tensor straight from the packed int array (no per-token Python objects: with ~200M tokens a list of lists
# would need many GB of RAM). Same chunks as before: starts 0, CONTEXT_LEN, 2*CONTEXT_LEN, ... while start < len - CONTEXT_LEN.
num_blocks = max(0, -(-(len(all_word_ids) - CONTEXT_LEN) // CONTEXT_LEN))
_flat = torch.frombuffer(all_word_ids, dtype=torch.int32) if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids, dtype=torch.int32)
data = _flat[:num_blocks * CONTEXT_LEN].view(num_blocks, CONTEXT_LEN).to(torch.long)
del _flat
print(f"Number of training chunks: {num_blocks}")

# Shuffle the chunks (fixed seed, so this is reproducible) BEFORE the
# train/val split below. The corpus is source-blocked -- all of one
# source's words, then the next -- so without this, the validation slice
# (a prefix of `data`) would be drawn entirely from whichever source comes
# first (train_acts.json), not proportionally from all 4 sources. Training
# itself doesn't need this shuffle (the training loop below already
# reshuffles chunk order every epoch via torch.randperm), but the
# validation split is fixed once up front, so it needs its own shuffle.
val_split_seed = torch.Generator().manual_seed(42)
shuffled_chunk_order = torch.randperm(len(data), generator=val_split_seed)
data = data[shuffled_chunk_order]

# hold out a slice for validation / local testing -- a non-overlapping
# slice of (now-shuffled) chunks, so no word is ever shared between
# train_data and val_data
num_val = max(1, int(0.01 * len(data)))
val_data = data[:num_val]
train_data = data[num_val:]
print(f"Training chunks: {len(train_data)}, validation chunks: {len(val_data)}")

Number of training chunks: 89724
Training chunks: 88827, validation chunks: 897


## 7. Building blocks for the research knobs

Before defining the model itself, here are the pieces `NORM_TYPE`,
`NORM_POSITION` and `OPTIMIZER_TYPE` (and `ATTENTION_TYPE`, for this
architecture) plug into. Every architecture in all 4 notebooks uses these
same building blocks, so results are comparable across notebooks.

In [11]:
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

### Optimizer with `OPTIMIZER_TYPE` support (AdamW / Muon)

**Muon** (Keller Jordan) keeps a momentum buffer like normal SGD, but
before applying the update it "orthogonalizes" it (a fast Newton-Schulz
iteration pushes every singular value of the update matrix towards 1).
This only makes sense for 2D+ weight matrices — biases, embeddings and
norm weights still use plain AdamW underneath. This is the standard Muon
recipe (and why `build_optimizer` below actually returns two optimizers
wrapped so the training loop can use them like one).

In [12]:
# Fused AdamW: PyTorch runs the whole AdamW update in a few fused GPU kernels instead of many small ones.
# Same math, slightly faster. It needs the parameters on a CUDA GPU, so it is off on CPU.
USE_FUSED_ADAMW = torch.cuda.is_available()


def zeropower_via_newton_schulz(update_matrix, steps=5, eps=1e-7):
    """Orthogonalize a matrix (push its singular values towards 1) with a
    fast iterative approximation. This is the core trick behind Muon."""
    a, b, c = 3.4445, -4.7750, 2.0315
    x = update_matrix.float()
    flipped = x.size(-2) > x.size(-1)
    if flipped:
        x = x.transpose(-2, -1)
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        a_term = x @ x.transpose(-2, -1)
        b_term = b * a_term + c * a_term @ a_term
        x = a * x + b_term @ x
    if flipped:
        x = x.transpose(-2, -1)
    return x


class Muon(torch.optim.Optimizer):
    """Simplified Muon optimizer. Only meant to be given 2D+ weight matrices."""

    def __init__(self, params, lr=0.02, momentum=0.95, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                buf.mul_(group["momentum"]).add_(p.grad)
                update = zeropower_via_newton_schulz(buf, steps=group["ns_steps"])
                scale = max(1.0, p.grad.size(-2) / p.grad.size(-1)) ** 0.5
                p.add_(update.reshape(p.shape).to(p.dtype), alpha=-group["lr"] * scale)


class CombinedOptimizer:
    """Wraps two optimizers so the training loop can call .zero_grad() /
    .step() on it just like a single normal optimizer."""

    def __init__(self, optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for opt in self.optimizers:
            opt.zero_grad()

    def step(self):
        for opt in self.optimizers:
            opt.step()

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state_dicts):
        for opt, state in zip(self.optimizers, state_dicts):
            opt.load_state_dict(state)


def build_optimizer(model, optimizer_type, lr):
    """optimizer_type is "adamw" or "muon"."""
    if optimizer_type == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, fused=USE_FUSED_ADAMW)

    if optimizer_type == "muon":
        muon_params, adamw_params = [], []
        for name, p in model.named_parameters():
            # muon only works well on 2D+ "matrix" weights; everything else
            # (biases, norm weights, embedding tables) stays on adamw
            if p.ndim >= 2 and "emb" not in name.lower():
                muon_params.append(p)
            else:
                adamw_params.append(p)
        muon_opt = Muon(muon_params, lr=lr * 30)   # muon typically wants a bigger lr than adamw
        adamw_opt = torch.optim.AdamW(adamw_params, lr=lr, fused=USE_FUSED_ADAMW)
        return CombinedOptimizer([muon_opt, adamw_opt])

    raise ValueError(f"unknown optimizer_type {optimizer_type}")

## 8. The model: Decoder-Only Transformer with GQA and a KV Cache

A **decoder-only Transformer with Grouped-Query Attention (GQA), a real KV
cache, and a choice of position encoding**. Training matches
`phase1_decoder_only.ipynb` with `ATTENTION_TYPE = "gqa"` (same layers, same
data, same optimizer) except for positions: this notebook defaults to
**RoPE** (`POSITIONAL_ENCODING = "rope"`, Section 8 cell); set it to
`"learned"` to get exactly the original model's architecture (same parameter
names/shapes). The tokenizer is the from-scratch Assignment-1 BPE (Section 5), so
checkpoints from the older notebooks (which used a library tokenizer) are not
compatible. What's new beyond that is **generation**.

### The problem this notebook fixes

In the attention-comparison runs (MHA vs GQA vs MQA) GQA finished in almost the
same wall-clock time as MHA. That is not a sign GQA "doesn't work" -- it is a
consequence of how the experiment was wired, for two separate reasons:

1. **Training.** Before calling `scaled_dot_product_attention`, the original
   code expanded GQA's K/V back to full width with `repeat_interleave`. The
   expensive part of attention (`QK^T`, softmax, `@V`, all O(seq_len^2)) then
   runs on full `n_heads`-wide tensors, so its cost is identical to MHA. GQA
   only shrinks the `k_proj`/`v_proj` weight matrices, a small slice of the
   total compute.
2. **Generation.** Every earlier notebook recomputes the *entire* growing
   sequence for each of the ~190 generated tokens, so there is no cache for GQA
   to shrink. GQA's design goal is precisely a smaller KV cache (Ainslie et
   al., 2023): fewer bytes to store and re-read on every decode step.

### What a KV cache is

Generation has two phases:

* **Prefill** -- run the whole prompt through the model once, in parallel
  (exactly the training-style forward pass), and *store* every layer's keys and
  values.
* **Decode** -- for each new token, compute Q, K, V for that **one** token only,
  append its K/V to the cache, and let the single new query attend to the cached
  keys/values. The prompt is never re-processed.

Per generated token the work drops from "a forward pass over ~500 tokens" to
"a forward pass over 1 token plus one attention read of the cache". Queries are
never cached (only the current token has one); the cache is pre-allocated once
per batch (as in `gpt-fast`) and written in place, so nothing is re-allocated or
concatenated per step.

### Why GQA is the right partner for it

Cache size per token = `2 (K and V) x n_layers x n_kv_heads x head_dim x bytes`.
With `n_heads = 12` and groups of 4, `n_kv_heads = 3`, so the cache is **4x
smaller than MHA's**. Decoding is *memory-bandwidth-bound* (one token of math,
but the whole cache has to be read from GPU memory every step), so fewer cache
bytes translates fairly directly into faster decode steps and larger batches.
The build cell prints the actual cache size for this model versus the MHA
equivalent.

### Reading the cache without un-doing GQA's savings

During decode the new token has 1 query position per query head. Instead of
expanding K/V 4x (which would copy and re-read a 4x-larger tensor every step and
throw away exactly what GQA saved), the 4 query heads that share a K/V head are
**folded into the query-length axis**: queries become `(batch, n_kv_heads,
group_size, head_dim)` and attend directly to the unexpanded cache
`(batch, n_kv_heads, kv_len, head_dim)`. No causal mask is needed (the new token
is last, so it may attend to everything cached). The result is mathematically
identical to expanding K/V -- checked numerically below and in the standalone
test -- while each cache byte is read once per K/V head instead of once per query
head.

Prefill and training use the ordinary parallel path
(`is_causal=True`), where `is_causal` lets PyTorch pick a fused kernel.

### What "flash" actually means on this hardware (honest notes)

* PyTorch's FlashAttention kernel needs **fp16/bf16** and an **sm80+** GPU
  (Ampere or newer). A Kaggle T4 is sm75, and the shared training loop runs in
  fp32, so during *training* SDPA uses the memory-efficient (or math) kernel, not
  FlashAttention. Generation here runs under fp16 autocast. The cell in Section
  9 prints which SDPA backends this session can really run, per dtype -- use
  that output, not assumptions, when the report says "flash".
* `enable_gqa=True` (native GQA in SDPA) is not always faster: a Hugging Face
  transformers issue documents it silently dispatching to the slow *math*
  kernel on sm<80 GPUs, where pre-expanding K/V was 1.6-1.8x faster. So the
  parallel path here **micro-benchmarks both variants on the actual GPU** and
  picks the faster one (`USE_NATIVE_GQA`). The decode path doesn't depend on
  `enable_gqa` at all (see the folding trick above).

### Positions: RoPE (default) or a learned table

**`"learned"`** -- a trainable `CONTEXT_LEN`-row table whose row `p` is added
to the token embedding at position `p` (GPT-2 style; what the original notebook
used). It only knows absolute positions `0 .. CONTEXT_LEN-1`. (Training feeds
`batch[:, :-1]`, so the very last row never receives a gradient; generation can
only touch it in the final slot before a re-prefill. RoPE has no such table.)

**`"rope"`** -- rotary position embeddings (RoFormer, Su et al. 2021). There is
no table. Inside every attention layer each query/key vector is split into
pairs of dimensions, and the pair `i` is rotated by the angle
`position x theta_i` with `theta_i = 10000^(-2i/head_dim)`. The code is the
same as in Assignment 3 (`build_rope_cache` / `rotate_half` / `apply_rope`, the
split-half layout where channel `i` pairs with channel `i + head_dim/2`; it was
checked to be bit-identical to that `rope.py`), only extended with per-row
position ids so the cache and left-padding work. Rotating both
sides means the dot product between a query at position `t` and a key at
position `p` depends only on `t - p`: **relative** position is built into the
attention score, at every layer, instead of being added once at the input.
It also removes the ~1.6M position-table parameters (2048 x 768).

**How RoPE and the cache fit together.** A key is rotated by its own position
exactly once, when it is written, and the cache stores the *already-rotated*
keys (values are never rotated). The new token's query is rotated at its own
position and then compared against the cache -- nothing cached is ever
re-touched. Position ids are per row (each row's first real token is
position 0, as in training), so left-padding does not shift anyone's angles.

### A design constraint worth stating in the report: the window

Cached keys/values are tied to the window they were computed in, so the cache
cannot *slide* the way the old recompute-everything loop did (which re-fed the
latest `CONTEXT_LEN` tokens every step). This holds for **both** position modes
-- with RoPE the position table is gone, but the model was still trained only on
2048-token windows (relative distances up to 2047) and the cache holds 2048
slots. A tempting shortcut with RoPE is a rolling cache that just drops the
oldest entry each step; StreamingLLM (Xiao et al. 2023) shows that this kind
of window attention over cached states breaks down once the earliest tokens
are evicted, because models park a lot of attention on the first tokens
("attention sinks"). So this notebook keeps the exact scheme:

* the prompt is cut to its last `PROMPT_KEEP = CONTEXT_LEN - CACHE_HEADROOM`
  tokens, leaving `CACHE_HEADROOM` (default 512) slots to generate into;
* if generation fills the cache, the last `PROMPT_KEEP` tokens (prompt +
  generated so far) are **re-prefilled** into a fresh cache -- recomputed from
  scratch, so the window always starts with a genuine first token -- one cheap
  parallel pass every ~`CACHE_HEADROOM` tokens instead of one per token.

So the model sees at most 1536 tokens of prompt (`PROMPT_KEEP`) rather than 2048.
A 512-word prompt is roughly 700-1000 BPE tokens, so in practice the *whole*
prompt fits, and 128 words of continuation (~200 tokens) fits in the headroom
without ever re-prefilling. Re-prefill only matters for unusually long prompts.
Tunable through `CACHE_HEADROOM`. A rolling cache that
keeps a few sink tokens (StreamingLLM-style) is a possible follow-up, but it
changes the math relative to the trained model and would be a separate
experiment.

**What RoPE is expected to change:** training-time modelling quality
(relative-position bias at every layer), not cache mechanics. Whether it helps
on this corpus and model size is exactly what running both modes tells you --
the checkpoint filename records the mode so the runs don't collide.

### Decoding: why not greedy, and what replaces it

Greedy decoding (always take the most likely token) tends to fall into loops on
small models -- the same phrase again and again -- because a repeated phrase makes
the model *more* confident about repeating it (Holtzman et al. 2019, arXiv
1904.09751). The knobs (`TEMPERATURE` is in Settings; the rest are at the top of the
generation cell):

| setting | meaning |
|---|---|
| `NO_REPEAT_NGRAM = 4` | the continuation may never repeat any 4-token sequence it already generated (generated tokens only, so prompt phrases can still be reused). Loops become impossible; everything else is untouched |
| `TEMPERATURE = 0.3, TOP_P = 0.9` | low-temperature nucleus sampling: draw from the smallest set of tokens covering 90 % of the probability, after sharpening by T = 0.3 -- close to the model's best guess, but not a deterministic loop |
| `REPETITION_PENALTY = 1.05-1.15` | tokens already generated get their logit divided (if positive) / multiplied (if negative). Blunt: it also penalises "the", "of", so keep it small; off (1.0) by default |
| `TEMPERATURE = 0` | plain greedy -- deterministic |
| `TEMPERATURE = 0.1, TOP_P = 0.95` | the course starter's setting |
| `GEN_SEED = 0` | seeds sampling at the start of every `generate_*` call, so a run is reproducible |

The defaults (T 0.3, top-p 0.9, 4-gram blocking) are a *starting point*; Section 15
runs a sweep over nine settings on held-out Supreme Court windows and prints
BERTScore / ROUGE-L / BLEU-4 together with a repetition rate, so the choice can be
made from numbers. Beam search is not implemented: it costs ~4x the compute at
beam width 4, needs the cache re-ordered every step, and likelihood-maximising
search is the same failure mode as greedy for open-ended text (Holtzman et al.).

### Batch size and compute at 2048 context

Attention cost grows with the square of the window, so a 2048 window is
noticeably slower per token than 640 (the attention share of the FLOPs rises),
and activation memory grows with `BATCH_SIZE x CONTEXT_LEN`. `BATCH_SIZE = 2`
keeps ~4k tokens per optimiser step -- about what 8 x 640 was -- and about the
same memory. The training loop is fp32 (no autocast) so it will be slower than an
Ampere GPU running bf16; if the run doesn't fit the time budget, reduce
`CONTEXT_LEN` (it is one line in Settings).

### Batching, padding and positions

Prompts are sorted by token length so most batches contain equal-length prompts
and need **no padding and no mask** (letting the fused kernels run in their
fast, mask-free mode). When a batch does need padding it is left-padded, each
row's position ids start at 0 at its first real token (as in training), and pad
keys are masked out.

### How this is verified (all inside the notebook, on the real trained weights)

1. **fp32 logit check**: cached decode logits vs. full-recompute logits over
   several steps -- max absolute difference is printed.
2. **Same-schedule uncached generation**: an equally-windowed reference that
   recomputes everything each step is timed against the cached version on the
   same prompts, and the fraction of identical outputs is printed. (Greedy
   decoding can flip on a near-tie between two tokens under fp16 rounding, so
   outputs are compared and reported, not asserted.)

### References

* Ainslie et al. 2023, *GQA: Training Generalized Multi-Query Transformer Models
  from Multi-Head Checkpoints* -- https://arxiv.org/abs/2305.13245
* Shazeer 2019, *Fast Transformer Decoding: One Write-Head is All You Need*
  (MQA) -- https://arxiv.org/abs/1911.02150
* Dao et al. 2022, *FlashAttention* -- https://arxiv.org/abs/2205.14135;
  Dao 2023, *FlashAttention-2* -- https://arxiv.org/abs/2307.08691
* Dao et al. 2023, *Flash-Decoding for long-context inference* (why decode with
  a single query underuses a GPU) --
  https://crfm.stanford.edu/2023/10/12/flashdecoding.html
* Kwon et al. 2023, *Efficient Memory Management for LLM Serving with
  PagedAttention* (KV cache memory as the serving bottleneck) --
  https://arxiv.org/abs/2309.06180
* PyTorch docs, `scaled_dot_product_attention` (`enable_gqa`, backend support) --
  https://docs.pytorch.org/docs/main/generated/torch.nn.functional.scaled_dot_product_attention.html
* `gpt-fast` (pre-allocated static `KVCache` written in place) --
  https://github.com/pytorch-labs/gpt-fast
* Hugging Face transformers PR #45776, on `enable_gqa=True` silently falling
  back to the math kernel on sm<80 -- https://github.com/huggingface/transformers/pull/45776
* Sebastian Raschka, *KV cache* (prefill/decode explanation) --
  https://sebastianraschka.com/faq/docs/kv-cache.html
* Su et al. 2021, *RoFormer: Enhanced Transformer with Rotary Position
  Embedding* (RoPE) -- https://arxiv.org/abs/2104.09864
* Xiao et al. 2023, *Efficient Streaming Language Models with Attention Sinks*
  (why a plain rolling cache breaks) -- https://arxiv.org/abs/2309.17453

In [13]:
# ---- Settings specific to this notebook ------------------------------
GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 9. Build the model

Sized so the parameter count lands in the 50M-300M range the brief
requires (printed below so you can check).

In [14]:
vocab_size = bpe_tokenizer.get_vocab_size()
model = CachedDecoderModel(vocab_size, d_model=768, n_heads=12, n_layers=8,
                           d_ff=3072, context_len=CONTEXT_LEN)
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
if not (50_000_000 <= num_params <= 300_000_000):
    print("WARNING: this is outside the required 50M-300M range! "
          "Change d_model / n_layers above.")

# ---- Checkpoint name records the positional-encoding choice ----------------
# A RoPE model has no position table, so its weights cannot load into a
# "learned" model or vice versa, and a model trained at one CONTEXT_LEN should not
# silently resume into a run at another; the filename records both.
print(f"Positional encoding: {POSITIONAL_ENCODING}   context length: {CONTEXT_LEN}")
# the checkpoint name also records the position mode, the context length and how much data
# the tokenizer + model were trained on (a different data size means a different tokenizer,
# so its checkpoint must never be resumed into another run)
DATA_TAG = f"{round(word_count / 1e6)}Mw" if word_count >= 1_000_000 else f"{word_count // 1000}kw"
# "a1bpe" = the from-scratch Assignment-1 BPE of Section 5; a checkpoint trained with any other tokenizer has different token ids.
CHECKPOINT_PATH = CHECKPOINT_PATH.replace("_checkpoint.pt", f"_{POSITIONAL_ENCODING}_ctx{CONTEXT_LEN}_{DATA_TAG}_a1bpe_{RUN_TAG}_checkpoint.pt")

# (The old "reuse the original notebook's checkpoint" option is gone: those checkpoints were trained
# with a different tokenizer, so their token ids would not line up with this notebook's own BPE.)

# ---- KV-cache memory: GQA (this model) vs the MHA equivalent ---------------
_attn0 = model.blocks[0].attn
_bytes_per_token_gqa = 2 * len(model.blocks) * _attn0.n_kv_heads * _attn0.head_dim * 2   # K and V, fp16
_bytes_per_token_mha = 2 * len(model.blocks) * _attn0.n_heads * _attn0.head_dim * 2
print(f"\nKV cache per sequence at {CONTEXT_LEN} tokens (fp16): "
      f"GQA {_bytes_per_token_gqa * CONTEXT_LEN / 1e6:.1f} MB "
      f"vs MHA {_bytes_per_token_mha * CONTEXT_LEN / 1e6:.1f} MB "
      f"({_attn0.n_heads // _attn0.n_kv_heads}x smaller)")


# ---- Which fused-attention kernels can this GPU / dtype actually run? -------
def probe_sdpa_backends():
    """Try each PyTorch SDPA backend on a small causal problem, in both fp32
    (what the shared training loop uses) and fp16 (what generation uses)."""
    if device != "cuda":
        print("SDPA backend probe: running on CPU, skipping (the CUDA kernels below only exist on a GPU).")
        return
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
    except Exception as e:
        print(f"SDPA backend probe unavailable in this torch version ({type(e).__name__}).")
        return
    print(f"GPU: {torch.cuda.get_device_name(0)}, compute capability {torch.cuda.get_device_capability(0)}")
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(2, 12, 256, 64, device=device, dtype=dtype)
        for name in ("FLASH_ATTENTION", "EFFICIENT_ATTENTION", "CUDNN_ATTENTION", "MATH"):
            backend = getattr(SDPBackend, name, None)
            if backend is None:
                continue
            try:
                with sdpa_kernel(backend):
                    F.scaled_dot_product_attention(q, q, q, is_causal=True)
                status = "available"
            except Exception as e:
                status = f"NOT available ({type(e).__name__})"
            print(f"  {str(dtype):15s} {name:20s} {status}")


# ---- Native GQA (enable_gqa) vs pre-expanded K/V: measure, don't guess ------
def pick_gqa_prefill_path():
    """Decides the global USE_NATIVE_GQA by timing both variants of the
    parallel attention call at the training shape, in fp32 and fp16. Native GQA
    is only chosen if it works AND is at least 10% faster in both dtypes."""
    global USE_NATIVE_GQA
    USE_NATIVE_GQA = False
    if device != "cuda" or _attn0.group_size == 1:
        print("Parallel attention path: pre-expanded K/V (repeat_interleave) -- no GPU benchmark on this device.")
        return
    seq_len, batch = CONTEXT_LEN - 1, BATCH_SIZE
    wins = []
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(batch, _attn0.n_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        k = torch.randn(batch, _attn0.n_kv_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        v = torch.randn_like(k)

        def run_expanded():
            ke = k.repeat_interleave(_attn0.group_size, dim=1)
            ve = v.repeat_interleave(_attn0.group_size, dim=1)
            return F.scaled_dot_product_attention(q, ke, ve, is_causal=True)

        def run_native():
            return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)

        timings = {}
        for label, fn in (("expanded", run_expanded), ("native", run_native)):
            try:
                for _ in range(3):
                    fn()
                torch.cuda.synchronize()
                start = torch.cuda.Event(enable_timing=True)
                stop = torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(20):
                    fn()
                stop.record()
                torch.cuda.synchronize()
                timings[label] = start.elapsed_time(stop) / 20
            except Exception as e:
                timings[label] = None
                print(f"  {label} attention failed in {dtype}: {type(e).__name__}")
        if timings["expanded"] is not None and timings["native"] is not None:
            print(f"  {str(dtype):15s} pre-expanded K/V: {timings['expanded']:.2f} ms   "
                  f"enable_gqa=True: {timings['native']:.2f} ms")
            wins.append(timings["native"] < 0.9 * timings["expanded"])
        else:
            wins.append(False)
    USE_NATIVE_GQA = all(wins)
    print(f"Parallel attention path chosen: "
          f"{'enable_gqa=True (native)' if USE_NATIVE_GQA else 'pre-expanded K/V (repeat_interleave)'}")


probe_sdpa_backends()
pick_gqa_prefill_path()
log_event("model_built", params=num_params, vocab_size=vocab_size, d_model=model.token_emb.embedding_dim, n_heads=model.blocks[0].attn.n_heads,
          n_kv_heads=model.blocks[0].attn.n_kv_heads, n_layers=len(model.blocks), d_ff=model.blocks[0].ff[0].out_features, positional_encoding=POSITIONAL_ENCODING, checkpoint=os.path.basename(CHECKPOINT_PATH))

Number of parameters: 111,095,872 (111.1M)
Positional encoding: rope   context length: 2048

KV cache per sequence at 2048 tokens (fp16): GQA 12.6 MB vs MHA 50.3 MB (4x smaller)
GPU: Tesla T4, compute capability (7, 5)
  torch.float32   FLASH_ATTENTION      NOT available (RuntimeError)


/tmp/ipykernel_23/378392225.py:58: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:960.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/378392225.py:58: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/378392225.py:58: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:962.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/378392225.py:58: UserWarning: Flash attention only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:339.)
  F.scaled_dot_pr

  torch.float32   EFFICIENT_ATTENTION  available
  torch.float32   CUDNN_ATTENTION      NOT available (RuntimeError)
  torch.float32   MATH                 available
  torch.float16   FLASH_ATTENTION      NOT available (RuntimeError)
  torch.float16   EFFICIENT_ATTENTION  available
  torch.float16   CUDNN_ATTENTION      NOT available (RuntimeError)
  torch.float16   MATH                 available


/tmp/ipykernel_23/378392225.py:58: UserWarning: cuDNN MHA only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:632.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)


  torch.float32   pre-expanded K/V: 7.92 ms   enable_gqa=True: 22.66 ms
  torch.float16   pre-expanded K/V: 1.13 ms   enable_gqa=True: 25.01 ms
Parallel attention path chosen: pre-expanded K/V (repeat_interleave)


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 10. Resume from a checkpoint if one already exists

If this is the first time you're running this notebook (in this mode), nothing to resume -- training starts
from scratch. Otherwise the checkpoint (local, or pulled from the Hugging Face repo) says where training stopped:
either at the end of an epoch, or **part-way through one** (Section 11 saves every `CKPT_EVERY_STEPS` batches). In the
second case training continues from that exact batch of that epoch; this works because each epoch's shuffle
order is seeded by the epoch number, so the same batches come in the same order after a restart.

In [15]:
optimizer = build_optimizer(model, OPTIMIZER_TYPE, LEARNING_RATE)
start_epoch = 0          # first epoch still to (finish) train
resume_batch = 0         # batches of start_epoch that are already done
resume_loss_sum = 0.0    # their summed loss (only for the running average that is printed)

try_pull_checkpoint_from_hub(CHECKPOINT_PATH)  # in case a fresh session/account has no local copy

if os.path.exists(CHECKPOINT_PATH):
    print("Found an existing checkpoint, resuming from it...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"] + 1           # "epoch" = last epoch that was COMPLETED (-1 = none yet)
    if checkpoint.get("in_progress_epoch") == start_epoch:
        resume_batch = checkpoint["next_batch"]
        resume_loss_sum = checkpoint.get("loss_sum", 0.0)
        print(f"Resuming in the middle of epoch {start_epoch}, at batch {resume_batch}")
    else:
        print(f"Resuming from the start of epoch {start_epoch}")
    log_event("resume", start_epoch=start_epoch, resume_batch=resume_batch)
    del checkpoint
else:
    print("No checkpoint found. Starting from scratch.")

No matching checkpoint on the Hub yet (RemoteEntryNotFoundError) -- starting fresh.
No checkpoint found. Starting from scratch.


## 11. Train the model

This is the main loop: show the model a batch of examples, see how wrong its predictions were (the "loss"), and
nudge its weights to do a little better next time.

**Checkpoints.** A checkpoint (weights + optimizer state + position in training) is written to disk every
`CKPT_EVERY_STEPS` batches and at the end of every epoch, and pushed to the Hugging Face repo every
`PUSH_EVERY_STEPS` batches and at the end of every epoch. If the Kaggle session dies (12-hour cap, crash), the next run
resumes from the last checkpoint (Section 10) instead of losing up to a whole epoch. The file is written to a temporary
name and then renamed, so a crash *during* saving cannot corrupt the previous checkpoint. The per-epoch shuffle is seeded
(`1000 + epoch`), which is what makes exact mid-epoch resumption possible. Each save/push of a ~1.3 GB file pauses
training for a short while, so the intervals are a trade-off between lost work and lost time.

In [16]:
import math

CKPT_EVERY_STEPS = 50 if QUICK_TEST else 2000     # save a checkpoint to disk every this many batches
PUSH_EVERY_STEPS = 100 if QUICK_TEST else 4000    # ...and push it to the Hugging Face repo every this many (multiple of the above)
assert PUSH_EVERY_STEPS % CKPT_EVERY_STEPS == 0

print(f"Training Decoder-Only Transformer with GQA and a KV Cache for {NUM_EPOCHS} epoch(s), QUICK_TEST={QUICK_TEST}, "
      f"NORM_TYPE={NORM_TYPE}, NORM_POSITION={NORM_POSITION}, "
      f"OPTIMIZER_TYPE={OPTIMIZER_TYPE}...")


def save_training_state(last_done_epoch, in_progress_epoch=None, next_batch=0, loss_sum=0.0):
    """last_done_epoch = last COMPLETED epoch (-1 if none). When part-way through an epoch, also record which
    epoch and which batch to continue from. Written atomically (temp file, then rename)."""
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": last_done_epoch}
    if in_progress_epoch is not None:
        state.update(in_progress_epoch=in_progress_epoch, next_batch=next_batch, loss_sum=loss_sum)
    tmp_path = CHECKPOINT_PATH + ".tmp"
    torch.save(state, tmp_path)
    os.replace(tmp_path, CHECKPOINT_PATH)


def push_state_and_log():
    try:
        push_checkpoint_to_hub(CHECKPOINT_PATH)
    except Exception as e:     # a failed upload must not kill a long training run; the local file is still there
        print(f"(checkpoint push failed: {type(e).__name__} -- local copy is fine, will retry at the next push)")
    sync_run_log()


batches_per_epoch = math.ceil(len(train_data) / BATCH_SIZE)
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    shuffled_order = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(1000 + epoch))
    if epoch == start_epoch and resume_batch > 0:
        num_batches, total_loss = resume_batch, resume_loss_sum      # continue part-way through this epoch
    else:
        num_batches, total_loss = 0, 0.0
    batches_at_session_start, t_session = num_batches, time.time()
    log_event("epoch_start", epoch=epoch, from_batch=num_batches, batches_per_epoch=batches_per_epoch)

    for start in range(num_batches * BATCH_SIZE, len(train_data), BATCH_SIZE):
        batch_positions = shuffled_order[start:start + BATCH_SIZE]
        batch = train_data[batch_positions].to(device)

        inputs = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(inputs)

        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        if num_batches % 100 == 0:
            rate = (num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session)
            eta_h = (batches_per_epoch - num_batches) / max(1e-9, rate) / 3600
            print(f"  epoch {epoch}, batch {num_batches}/{batches_per_epoch}: loss = {loss.item():.3f}  "
                  f"({rate:.2f} batch/s, ~{eta_h:.1f} h left in this epoch)")

        if num_batches % CKPT_EVERY_STEPS == 0 and num_batches < batches_per_epoch:
            save_training_state(epoch - 1, in_progress_epoch=epoch, next_batch=num_batches, loss_sum=total_loss)
            log_event("checkpoint_mid_epoch", epoch=epoch, batch=num_batches, running_avg_loss=round(total_loss / num_batches, 4),
                      batches_per_sec=round((num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session), 3))
            if num_batches % PUSH_EVERY_STEPS == 0:
                push_state_and_log()

    avg_loss = total_loss / max(1, num_batches)
    print(f"Epoch {epoch} finished. Average loss: {avg_loss:.3f}")

    save_training_state(epoch)
    print("Checkpoint saved.")
    log_event("epoch_end", epoch=epoch, avg_loss=round(avg_loss, 4), batches=num_batches,
              session_minutes=round((time.time() - t_session) / 60, 1))
    push_state_and_log()

print("Training complete!")

Training Decoder-Only Transformer with GQA and a KV Cache for 1 epoch(s), QUICK_TEST=False, NORM_TYPE=layernorm, NORM_POSITION=pre, OPTIMIZER_TYPE=adamw...


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


  epoch 0, batch 100/44414: loss = 7.796  (1.29 batch/s, ~9.5 h left in this epoch)
  epoch 0, batch 200/44414: loss = 6.598  (1.22 batch/s, ~10.1 h left in this epoch)
  epoch 0, batch 300/44414: loss = 7.104  (1.19 batch/s, ~10.3 h left in this epoch)
  epoch 0, batch 400/44414: loss = 5.948  (1.18 batch/s, ~10.3 h left in this epoch)
  epoch 0, batch 500/44414: loss = 6.085  (1.18 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 600/44414: loss = 5.498  (1.17 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 700/44414: loss = 5.581  (1.17 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 800/44414: loss = 7.014  (1.16 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 900/44414: loss = 6.058  (1.16 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 1000/44414: loss = 5.793  (1.16 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 1100/44414: loss = 5.453  (1.16 batch/s, ~10.4 h left in this epoch)
  epoch 0, batch 1200/44414: loss = 6.240  (1.16 batch/s, ~10.4 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt to https://huggingface.co/kavincg23/cs781-checkpoints/blob/main/phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt
  epoch 0, batch 4100/44414: loss = 4.424  (1.14 batch/s, ~9.8 h left in this epoch)
  epoch 0, batch 4200/44414: loss = 4.821  (1.14 batch/s, ~9.8 h left in this epoch)
  epoch 0, batch 4300/44414: loss = 4.114  (1.14 batch/s, ~9.7 h left in this epoch)
  epoch 0, batch 4400/44414: loss = 4.681  (1.14 batch/s, ~9.7 h left in this epoch)
  epoch 0, batch 4500/44414: loss = 4.190  (1.14 batch/s, ~9.7 h left in this epoch)
  epoch 0, batch 4600/44414: loss = 4.597  (1.14 batch/s, ~9.7 h left in this epoch)
  epoch 0, batch 4700/44414: loss = 4.339  (1.14 batch/s, ~9.6 h left in this epoch)
  epoch 0, batch 4800/44414: loss = 4.107  (1.14 batch/s, ~9.6 h left in this epoch)
  epoch 0, batch 4900/44414: loss = 4

... (29 outputs left out)

Pushed phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt to https://huggingface.co/kavincg23/cs781-checkpoints/blob/main/phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt
  epoch 0, batch 44100/44414: loss = 2.724  (1.14 batch/s, ~0.1 h left in this epoch)
  epoch 0, batch 44200/44414: loss = 2.409  (1.14 batch/s, ~0.1 h left in this epoch)
  epoch 0, batch 44300/44414: loss = 3.029  (1.14 batch/s, ~0.0 h left in this epoch)
  epoch 0, batch 44400/44414: loss = 3.218  (1.14 batch/s, ~0.0 h left in this epoch)
Epoch 0 finished. Average loss: 3.777
Checkpoint saved.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt to https://huggingface.co/kavincg23/cs781-checkpoints/blob/main/phase1_decoder_only_gqa_kvcache_full_layernorm_pre_adamw_rope_ctx2048_160Mw_a1bpe_160Mw_1ep_checkpoint.pt
Training complete!


## 12. Check validation loss

This tells you how the model does on chunks it did NOT train on — a much
more honest signal than the training loss above.

In [17]:
model.eval()
total_val_loss = 0.0
num_val_batches = 0

with torch.no_grad():
    for start in range(0, len(val_data), BATCH_SIZE):
        batch = val_data[start:start + BATCH_SIZE].to(device)
        if len(batch) == 0:
            continue
        inputs = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(inputs)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        total_val_loss += loss.item()
        num_val_batches += 1

avg_val_loss = total_val_loss / max(1, num_val_batches)
print(f"Validation loss: {avg_val_loss:.3f}")
print(f"Validation perplexity: {2.718281828 ** avg_val_loss:.1f}  (lower is better)")
log_event("validation_1pct", loss=round(avg_val_loss, 4), perplexity=round(2.718281828 ** avg_val_loss, 2), chunks=len(val_data))

Validation loss: 3.077
Validation perplexity: 21.7  (lower is better)


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 13. Generate the 128-word continuations for the test set

The competition wants exactly 128 words for every test prompt, containing
ONLY the generated continuation (not the original prompt).

## 13. Generate the 128-word continuations for the test set

The competition wants exactly 128 words for every test prompt, containing
ONLY the generated continuation (not the original prompt).

**Generation here uses the KV cache** (see Section 8 for the full story):
one *prefill* pass over each prompt window, then one-token *decode* steps that
only read the cached keys/values. Two reference/verification pieces sit next
to it:

* `generate_continuations_batch_uncached` -- the same windowing schedule, but
  recomputing the whole window every step (no cache). It exists purely to
  check the cached version against and to time it.
* the cell after the generation functions runs an **fp32 logit-equivalence
  check** and a **timing + output-agreement benchmark**; both print their
  results for the report.

Prompts are sorted by token length before batching so most batches need no
padding (and therefore no attention mask); results are put back in the
original order. Just like the other BPE notebooks, each sequence keeps
generating until it has STRICTLY MORE than `num_words` words, so the
`num_words`-th word is fully finished and not a subword fragment.

**Decoding** is no longer plain greedy (it looped): the defaults are low-temperature nucleus
sampling plus 4-gram repeat blocking, seeded so a run is reproducible. See the knobs at the top of
the next cell and the sweep in Section 15, which scores several settings so you can choose.

In [18]:
PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0
PROMPT_KEEP = CONTEXT_LEN - CACHE_HEADROOM          # max prompt-window tokens (see Section 8)
CACHE_DTYPE = torch.float16 if device == "cuda" else torch.float32   # cache stored in the dtype generation computes in
assert 0 < PROMPT_KEEP < CONTEXT_LEN, "CACHE_HEADROOM must be between 1 and CONTEXT_LEN - 1"

# ---- Decoding knobs (TEMPERATURE itself lives in the Settings cell) -----------
# Plain greedy decoding (TEMPERATURE = 0) tends to fall into loops on small models,
# so the defaults here are a starting point that fights repetition; Step 15 has a
# sweep that scores several settings on held-out Supreme Court examples so you can
# choose from numbers. For plain greedy: TEMPERATURE = 0, NO_REPEAT_NGRAM = 0.
# To mimic the official starter: TEMPERATURE = 0.1 (Settings cell), TOP_P = 0.95.
TOP_P = 0.9                # nucleus sampling, used only when TEMPERATURE > 0: sample from the
                           # smallest set of tokens whose probabilities add up to TOP_P. 1.0 = off.
REPETITION_PENALTY = 1.0   # > 1 makes tokens this continuation has ALREADY generated less likely
                           # (CTRL / Hugging Face style: positive logits divided, negative ones
                           # multiplied). Applies to generated tokens only, not the prompt, so
                           # names and terms from the prompt can still be reused. 1.0 = off.
                           # Note it hits ALL reused tokens (incl. "the", "of"), so keep it small (1.05-1.15).
NO_REPEAT_NGRAM = 4        # never let the continuation repeat any sequence of this many TOKENS
                           # (Hugging Face `no_repeat_ngram_size`, on generated tokens only). More
                           # surgical than a penalty: it only blocks exact repeats, so loops die but
                           # ordinary words can be reused. 0 = off; 4 tokens is roughly 3 words.
GEN_SEED = 0               # sampling seed, set at the start of every generate_* call so a run is
                           # reproducible (only matters when TEMPERATURE > 0). None = don't seed.


def fp16_autocast():
    # T4 GPUs have dedicated fp16 tensor cores; no gradients here, so none of the
    # usual mixed-precision-training complications apply. No-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda"))


class NoRepeatNgram:
    """Bans any token that would complete an n-token sequence this continuation has
    already generated (like Hugging Face's no_repeat_ngram_size, but looking only at
    the GENERATED tokens, so words and phrases can still be copied from the prompt)."""

    def __init__(self, batch_size, n):
        assert n >= 2, "NO_REPEAT_NGRAM must be 0 (off) or >= 2"
        self.n = n
        self.tokens = [[] for _ in range(batch_size)]
        self.follow = [dict() for _ in range(batch_size)]   # (n-1)-token prefix -> tokens that followed it

    def _prefix(self, toks):
        return tuple(toks[len(toks) - (self.n - 1):])

    def banned(self, vocab_size):
        """(batch, vocab_size) bool mask: True = this token would repeat an n-gram."""
        mask = torch.zeros(len(self.tokens), vocab_size, dtype=torch.bool, device=device)
        for row, toks in enumerate(self.tokens):
            if len(toks) >= self.n - 1:
                blocked = self.follow[row].get(self._prefix(toks))
                if blocked:
                    mask[row, list(blocked)] = True
        return mask

    def update(self, next_list, done):
        """Record the token each still-generating row just produced."""
        for row, token in enumerate(next_list):
            if done[row]:
                continue
            toks = self.tokens[row]
            if len(toks) >= self.n - 1:
                self.follow[row].setdefault(self._prefix(toks), set()).add(token)
            toks.append(token)


def sample_next_word_ids_batch(logits_batch, temperature, seen=None, banned=None):
    """Pick the next token for every row. logits_batch: (batch, vocab_size).
    seen: optional (batch, vocab_size) bool -- tokens each row has already
    generated (only used when REPETITION_PENALTY != 1). banned: optional
    (batch, vocab_size) bool -- tokens that may not be picked (n-gram blocking).
    Returns (batch,) ids. Order: ban n-gram repeats -> repetition penalty ->
    (temperature 0: argmax) -> temperature -> top-p filter -> sample."""
    if banned is not None:
        logits_batch = logits_batch.masked_fill(banned, float("-inf"))
    if seen is not None and REPETITION_PENALTY != 1.0:
        penalised = torch.where(logits_batch > 0, logits_batch / REPETITION_PENALTY,
                                logits_batch * REPETITION_PENALTY)
        logits_batch = torch.where(seen, penalised, logits_batch)
    if temperature == 0:
        return logits_batch.argmax(dim=-1)
    logits_batch = logits_batch / temperature
    if TOP_P < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits_batch, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        # drop every token whose predecessors already add up to more than TOP_P
        # (so the single most likely token is always kept)
        drop = (sorted_probs.cumsum(dim=-1) - sorted_probs) > TOP_P
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits_batch = torch.full_like(logits_batch, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    probabilities = F.softmax(logits_batch, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).squeeze(-1)


def _track_seen(seen, next_ids, done):
    """Record the tokens just generated for rows that were still generating."""
    if seen is None:
        return
    rows = torch.arange(len(done), device=next_ids.device)
    still_generating = torch.tensor([not d for d in done], device=next_ids.device)
    seen[rows, next_ids] = seen[rows, next_ids] | still_generating


def _pad_windows(windows):
    """Left-pad a list of token-id lists to a common length. Returns the padded
    id tensor, the per-row pad counts (list), and the pad counts as a tensor."""
    width = max(len(w) for w in windows)
    pad_lens = [width - len(w) for w in windows]
    x = torch.tensor([[PAD_ID] * pad_lens[i] + windows[i] for i in range(len(windows))],
                     dtype=torch.long, device=device)
    return x, pad_lens, torch.tensor(pad_lens, dtype=torch.long, device=device)


def _prefill_from_history(history, cache):
    """(Re)build the cache from the last PROMPT_KEEP tokens of each row's
    history. Returns (logits for the next token, pad tensor, has_padding)."""
    x, pad_lens, pad_t = _pad_windows([h[-PROMPT_KEEP:] for h in history])
    logits = model.prefill(x, pad_lens, cache)
    return logits, pad_t, any(p > 0 for p in pad_lens)


def _finish(results, batch_indices, generated, num_words):
    for row, original_index in enumerate(batch_indices):
        words = decode(generated[row]).split()
        words = words[:num_words]
        while len(words) < num_words:   # safety net (should basically never happen)
            words.append("<unk>")
        results[original_index] = " ".join(words)


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=64, verbose=True):
    """KV-cached generation. Lower batch_size if you hit a CUDA out-of-memory error."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))   # similar lengths -> little/no padding
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6   # safety cap so this can never loop forever
    attn0 = model.blocks[0].attn

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]     # prompt window + everything generated so far
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz
        cache = KVCache(len(model.blocks), bsz, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim,
                        device, CACHE_DTYPE)

        with torch.no_grad(), fp16_autocast():
            logits, pad_t, has_padding = _prefill_from_history(history, cache)      # PREFILL
            seen = torch.zeros_like(logits, dtype=torch.bool) if REPETITION_PENALTY != 1.0 else None
            ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
            for _ in range(max_new_tokens):
                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()          # one GPU->CPU sync per step, not one per row
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if cache.length >= cache.max_len:
                    # cache full: re-prefill the most recent PROMPT_KEEP tokens
                    logits, pad_t, has_padding = _prefill_from_history(history, cache)
                else:
                    logits = model.decode_step(next_ids, pad_t, has_padding, cache)   # DECODE

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuations_batch_uncached(prompt_texts, num_words=128, batch_size=64, verbose=False):
    """REFERENCE: the same windowing schedule as the cached generator (same
    prompt cut, same re-window points), but recomputing the whole window with
    the ordinary parallel forward pass on every step -- no cache. Only used to
    check and time the cached version."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz

        def new_window():
            starts = [len(h) - min(len(h), PROMPT_KEEP) for h in history]
            return starts, max(len(h) - s for h, s in zip(history, starts))

        starts, width = new_window()
        seen = None
        ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
        with torch.no_grad(), fp16_autocast():
            for _ in range(max_new_tokens):
                x, pad_lens, pad_t = _pad_windows([h[s:] for h, s in zip(history, starts)])
                arange = torch.arange(x.shape[1], device=device).unsqueeze(0)
                position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)
                mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
                logits = model(x, key_padding_mask=mask, position_ids=position_ids, last_position_only=True)
                if seen is None and REPETITION_PENALTY != 1.0:
                    seen = torch.zeros_like(logits, dtype=torch.bool)

                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if width >= CONTEXT_LEN:
                    starts, width = new_window()     # mirrors "cache full -> re-prefill"
                else:
                    width += 1

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  (uncached) generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above."""
    return generate_continuations_batch([prompt_text], num_words=num_words,
                                         batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
not possible to accept the plea of delay and laches in filing the writ petition and the delay in filing the review petition was not condoned by the writ court. The delay in filing writ petition was not condoned. The delay in preferring the writ petition was condoned by the Division Bench of this Court in the case of [Municipal Corporation of Greater Bombay Vs. A. K. Rajappa & Ors. Vs. State of Bombay & Ors.] (1985) 2 SCC 736. In the case of State of Gujarat Vs. Raja Ram & Ors. (1985) 2 SCR 626, this Court held as under: "Delay in filing writ petition, is not maintainable unless the delay is condoned by the Court in filing the petition or in the alternative for condonation of

Word count: 128 (should be 128)


### Verify and time the KV cache (results for the report)

1. **fp32 equivalence check** (no autocast): on the first few test prompts,
   compare the logits the cached decode produces against a full recompute of
   the same window, for several consecutive steps. The maximum absolute
   difference should be tiny (float rounding only) -- this is the direct
   evidence that caching changes *speed, not the function being computed*.
2. **Timing and output agreement**: generate the same prompts with the cached
   generator and with the uncached reference, and print the wall-clock times,
   the speedup, and how many outputs match exactly. Under fp16 the two paths
   round differently, so a greedy decode can flip on a near-tie; a few
   mismatches late in a text are expected and are reported, not hidden.

Set `RUN_CACHE_BENCHMARK = False` to skip this cell.

In [19]:
RUN_CACHE_BENCHMARK = True
BENCH_PROMPTS = 8 if QUICK_TEST else 16

if RUN_CACHE_BENCHMARK:
    import time
    model.eval()
    bench_texts = list(test_prompts.values())[:BENCH_PROMPTS]
    attn0 = model.blocks[0].attn

    # ---- 1) fp32 numerical equivalence: cached decode vs full recompute ----
    n_check_steps = min(8, CACHE_HEADROOM)
    with torch.no_grad():
        windows = [encode(t)[-PROMPT_KEEP:] for t in bench_texts[:4]]
        x, pad_lens, pad_t = _pad_windows(windows)
        has_padding = any(p > 0 for p in pad_lens)
        cache32 = KVCache(len(model.blocks), len(windows), attn0.n_kv_heads, CONTEXT_LEN,
                          attn0.head_dim, device, torch.float32)
        cached_logits = model.prefill(x, pad_lens, cache32)
        max_diff = 0.0
        for step in range(n_check_steps):
            xs, pls, pts = _pad_windows(windows)
            ar = torch.arange(xs.shape[1], device=device).unsqueeze(0)
            pos_ids = (ar - pts.unsqueeze(1)).clamp(min=0)
            msk = (ar >= pts.unsqueeze(1)) if any(p > 0 for p in pls) else None
            full_logits = model(xs, key_padding_mask=msk, position_ids=pos_ids, last_position_only=True)
            max_diff = max(max_diff, (cached_logits - full_logits).abs().max().item())
            next_ids = cached_logits.argmax(dim=-1)
            for i in range(len(windows)):
                windows[i].append(next_ids[i].item())
            cached_logits = model.decode_step(next_ids, pad_t, has_padding, cache32)
    print(f"fp32 check over {n_check_steps} decode steps: max |cached logits - recomputed logits| = {max_diff:.2e}")

    # ---- 2) timing + output agreement (fp16 on GPU) ----
    def timed(fn):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        out = fn()
        if device == "cuda":
            torch.cuda.synchronize()
        return out, time.time() - t0

    # Compare the two generators under GREEDY decoding (random sampling would make identical
    # outputs impossible to expect); the penalty / n-gram settings stay as configured.
    _saved_temperature = TEMPERATURE
    TEMPERATURE = 0.0
    generate_continuations_batch(bench_texts[:1], num_words=4, batch_size=1, verbose=False)   # warm-up (kernel init)
    cached_out, t_cached = timed(lambda: generate_continuations_batch(
        bench_texts, num_words=128, batch_size=BENCH_PROMPTS, verbose=False))
    uncached_out, t_uncached = timed(lambda: generate_continuations_batch_uncached(
        bench_texts, num_words=128, batch_size=BENCH_PROMPTS, verbose=False))
    TEMPERATURE = _saved_temperature

    identical = sum(a == b for a, b in zip(cached_out, uncached_out))
    print(f"{len(bench_texts)} prompts x 128 words:  cached {t_cached:.1f}s   "
          f"uncached (same windowing, full recompute) {t_uncached:.1f}s   "
          f"-> {t_uncached / max(t_cached, 1e-9):.1f}x faster with the KV cache")
    print(f"identical outputs: {identical} / {len(bench_texts)}")

    demo_cache = KVCache(len(model.blocks), 64, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim, device, CACHE_DTYPE)
    print(f"KV cache memory for a batch of 64 (GEN_BATCH_SIZE): {demo_cache.num_bytes() / 1e6:.0f} MB")
    del demo_cache

fp32 check over 8 decode steps: max |cached logits - recomputed logits| = 9.54e-06
16 prompts x 128 words:  cached 1.5s   uncached (same windowing, full recompute) 20.4s   -> 13.5x faster with the KV cache
identical outputs: 15 / 16
KV cache memory for a batch of 64 (GEN_BATCH_SIZE): 805 MB


In [20]:
print("Generating continuations for all test prompts...")
test_ids = list(test_prompts.keys())
test_texts = [test_prompts[doc_id] for doc_id in test_ids]

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
generated_texts = generate_continuations_batch(test_texts, num_words=128,
                                                batch_size=GEN_BATCH_SIZE)
predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, generated_texts)}

prediction_path = "/kaggle/working/prediction.json"
with open(prediction_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved {len(predictions)} predictions to {prediction_path}")

Generating continuations for all test prompts...
  generated 64 / 1000
  generated 128 / 1000
  generated 192 / 1000
  generated 256 / 1000
  generated 320 / 1000
  generated 384 / 1000
  generated 448 / 1000
  generated 512 / 1000
  generated 576 / 1000
  generated 640 / 1000
  generated 704 / 1000
  generated 768 / 1000
  generated 832 / 1000
  generated 896 / 1000
  generated 960 / 1000
  generated 1000 / 1000
Saved 1000 predictions to /kaggle/working/prediction.json


## 14. Check everything, then package `submission.zip`

Codabench needs `prediction.json` to sit at the ROOT of the zip file (not
inside a folder), and every official test ID must be present with no
extras. This cell checks both before zipping.

In [21]:
official_ids = set(test_prompts.keys())
predicted_ids = set(predictions.keys())

missing_ids = official_ids - predicted_ids
extra_ids = predicted_ids - official_ids

if missing_ids:
    print("PROBLEM: missing predictions for IDs:", missing_ids)
if extra_ids:
    print("PROBLEM: predictions include unknown IDs:", extra_ids)

wrong_word_count_ids = [doc_id for doc_id, pred in predictions.items()
                        if len(pred["text"].split()) != 128]
if wrong_word_count_ids:
    print("PROBLEM: these IDs do not have exactly 128 words:", wrong_word_count_ids)

if not missing_ids and not extra_ids and not wrong_word_count_ids:
    print("All checks passed: correct IDs, correct word counts.")

    submission_path = "/kaggle/working/submission.zip"
    with zipfile.ZipFile(submission_path, "w") as zip_file:
        # arcname="prediction.json" puts the file at the ROOT of the zip,
        # which is exactly what Codabench requires
        zip_file.write(prediction_path, arcname="prediction.json")

    print(f"Saved {submission_path}")
    print("If QUICK_TEST is still True, this is just a pipeline check -- "
          "don't submit it. Set QUICK_TEST = False, re-run the notebook, "
          "and package/submit THAT.")
    print("Download from the 'Output' section on the right side of this "
          "Kaggle notebook (you may need to click 'Save Version' first), "
          "then upload it to Codabench.")
else:
    print("Fix the problems above before packaging the submission.")

log_event("submission_zip_step", n_predictions=len(predictions), temperature=TEMPERATURE, top_p=TOP_P,
          repetition_penalty=REPETITION_PENALTY, no_repeat_ngram=NO_REPEAT_NGRAM, gen_seed=GEN_SEED)
sync_run_log()

All checks passed: correct IDs, correct word counts.
Saved /kaggle/working/submission.zip
If QUICK_TEST is still True, this is just a pipeline check -- don't submit it. Set QUICK_TEST = False, re-run the notebook, and package/submit THAT.
Download from the 'Output' section on the right side of this Kaggle notebook (you may need to click 'Save Version' first), then upload it to Codabench.


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 15. Test-like evaluation (held-out documents) and the decoding sweep

**Why this replaces the old validation.** The 1 % validation loss in Section 12 is a random slice of
*chunks of the training documents*: a chunk's neighbours from the same judgment are in the training set,
and the slice mixes the four sources evenly. The test set is different -- 512-word windows cut out of
the middle of *whole judgments that the model has never seen*, mostly Supreme Court. So, while collecting
the data (Section 6), **whole documents were held out per source** (never trained on, never used to fit
the tokenizer) and written to `heldout_<source>.txt`. Three things are measured on them:

1. **Held-out perplexity per source** -- one random window of up to `CONTEXT_LEN` tokens from each
   held-out document. Shows *where* the model is strong or weak (e.g. Acts vs Supreme Court).
2. **Test-prompt perplexity** -- perplexity on the 512-word test prompts themselves (their text is
   given to us; only the continuations are secret). This is the closest thing to the real test text.
3. **Local generation scores** -- 512-word prompt + 128-word reference cut from a random position
   inside held-out Supreme Court documents (so the prompt is a *mid-judgment window*, like the test
   prompts, and the reference is genuinely the next 128 words). Scored with BERTScore F1, ROUGE-L,
   BLEU-4 and a repetition rate (share of 4-word sequences that already occurred earlier in the same
   continuation).

(The old local eval decoded the *shuffled* validation chunks back-to-back, so many of its prompt/reference
windows straddled two unrelated chunks; that bug is gone.)

Perplexity is per BPE *token*, so compare it across models/runs that use the same tokenizer, and across
sources within a run only loosely (the tokens-per-word ratio differs by source).

Held-out files only exist when `QUICK_TEST = False`; in QUICK_TEST the cells below skip themselves.

In [22]:
import math, random

EVAL_SEED = 1234                 # fixed: every run scores the same windows
PPL_WINDOWS_PER_SOURCE = 200     # held-out documents scored per source (one window each)
TEST_PPL_PROMPTS = 300           # how many test prompts to score (seeded subset)
model.eval()


def window_nll(token_ids):
    """Summed next-token cross-entropy (nats) and number of predicted tokens for one window."""
    ids = token_ids[:CONTEXT_LEN]
    if len(ids) < 2:
        return 0.0, 0
    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([ids[1:]], dtype=torch.long, device=device)
    with torch.no_grad():
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="sum")
    return loss.item(), y.numel()


ppl_results = {}
if not HELDOUT_FILES:
    print("No held-out documents on disk (QUICK_TEST mode) -- skipping held-out perplexity.")
else:
    rng = random.Random(EVAL_SEED)
    for source, path in HELDOUT_FILES.items():
        with open(path, "r", encoding="utf-8") as f:
            docs = [line.strip() for line in f if line.strip()]
        rng.shuffle(docs)
        total_nll, total_tokens = 0.0, 0
        for doc in docs[:PPL_WINDOWS_PER_SOURCE]:
            ids = encode(doc)
            start = rng.randint(0, max(0, len(ids) - CONTEXT_LEN))
            nll, count = window_nll(ids[start:start + CONTEXT_LEN])
            total_nll += nll
            total_tokens += count
        if total_tokens:
            ppl_results[source] = math.exp(total_nll / total_tokens)
            print(f"held-out {source:14s}: perplexity {ppl_results[source]:8.2f}   "
                  f"({min(len(docs), PPL_WINDOWS_PER_SOURCE)} documents, {total_tokens:,} tokens)")

# perplexity on the actual test prompts (text only; the continuations are secret)
rng = random.Random(EVAL_SEED)
prompt_ids = list(test_prompts.keys())
rng.shuffle(prompt_ids)
total_nll, total_tokens = 0.0, 0
for doc_id in prompt_ids[:TEST_PPL_PROMPTS]:
    nll, count = window_nll(encode(test_prompts[doc_id]))
    total_nll += nll
    total_tokens += count
if total_tokens:
    ppl_results["test_prompts"] = math.exp(total_nll / total_tokens)
    print(f"test prompts        : perplexity {ppl_results['test_prompts']:8.2f}   "
          f"({min(len(prompt_ids), TEST_PPL_PROMPTS)} prompts, {total_tokens:,} tokens)")

if "avg_val_loss" in globals():
    print(f"for reference, the 1% mixed-chunk validation perplexity (Section 12): {math.exp(avg_val_loss):.2f}")

log_event("heldout_perplexity", results=ppl_results)

held-out acts          : perplexity    15.06   (140 documents, 258,932 tokens)
held-out high_courts   : perplexity    20.05   (200 documents, 297,705 tokens)
held-out supreme_court : perplexity    27.03   (200 documents, 366,706 tokens)
held-out tribunals     : perplexity    22.54   (200 documents, 302,454 tokens)
test prompts        : perplexity    30.78   (300 prompts, 167,735 tokens)
for reference, the 1% mixed-chunk validation perplexity (Section 12): 21.69


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


### Local generation scores on held-out Supreme Court windows

Builds up to `LOCAL_EVAL_EXAMPLES` (prompt, reference) pairs -- one random 640-word window per held-out
Supreme Court document, 512 words as the prompt and the next 128 as the reference -- and scores the
model's continuations with the **current decoding settings** (`TEMPERATURE`, `TOP_P`,
`REPETITION_PENALTY`, `NO_REPEAT_NGRAM`, `GEN_SEED`). BERTScore uses `roberta-large`, so the first run
downloads it.

In [23]:
!pip install -q bert-score rouge-score sacrebleu

from bert_score import BERTScorer
from rouge_score import rouge_scorer
import sacrebleu

LOCAL_EVAL_EXAMPLES = 200
local_prompts, local_references = [], []
sc_path = HELDOUT_FILES.get("supreme_court")
if sc_path:
    rng = random.Random(EVAL_SEED)
    with open(sc_path, "r", encoding="utf-8") as f:
        sc_docs = [line.split() for line in f]
    sc_docs = [d for d in sc_docs if len(d) >= 640]
    rng.shuffle(sc_docs)
    for words in sc_docs[:LOCAL_EVAL_EXAMPLES]:
        start = rng.randint(0, len(words) - 640)         # random position inside the judgment
        window = words[start:start + 640]
        local_prompts.append(" ".join(window[:512]))
        local_references.append(" ".join(window[512:]))
print(f"Built {len(local_prompts)} local evaluation examples from held-out Supreme Court documents")

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
_bert_scorer, _rouge = None, rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def repeated_4gram_rate(texts):
    """Share of word 4-grams that already occurred earlier in the SAME continuation
    (0 = no repetition; loops push it towards 1)."""
    repeated = total = 0
    for text in texts:
        words, seen = text.split(), set()
        for i in range(len(words) - 3):
            gram = tuple(words[i:i + 4])
            total += 1
            repeated += gram in seen
            seen.add(gram)
    return repeated / max(1, total)


def score_local(predictions, references):
    global _bert_scorer
    if _bert_scorer is None:
        _bert_scorer = BERTScorer(lang="en", batch_size=16)
    _, _, f1 = _bert_scorer.score(predictions, references)
    rouge_l = [_rouge.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]
    return {"BERTScore F1": f1.mean().item(),
            "ROUGE-L": sum(rouge_l) / len(rouge_l),
            "BLEU-4": sacrebleu.corpus_bleu(predictions, [references]).score,
            "repeated 4-grams": repeated_4gram_rate(predictions)}


if not local_prompts:
    print("Skipping local evaluation -- no held-out Supreme Court documents (QUICK_TEST mode?).")
else:
    local_predictions = generate_continuations_batch(local_prompts, num_words=128,
                                                       batch_size=GEN_BATCH_SIZE, verbose=False)
    current_scores = score_local(local_predictions, local_references)
    print(f"Current decoding: TEMPERATURE={TEMPERATURE}, TOP_P={TOP_P}, REPETITION_PENALTY={REPETITION_PENALTY}, "
          f"NO_REPEAT_NGRAM={NO_REPEAT_NGRAM}")
    for name, value in current_scores.items():
        print(f"  {name:18s}: {value:.4f}")
    print(f"  (reference text itself: repeated 4-grams = {repeated_4gram_rate(local_references):.4f})")

if local_prompts:
    log_event("local_eval", temperature=TEMPERATURE, top_p=TOP_P, repetition_penalty=REPETITION_PENALTY,
              no_repeat_ngram=NO_REPEAT_NGRAM, examples=len(local_prompts), scores=current_scores)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.0 MB/s eta 0:00:00
Built 200 local evaluation examples from held-out Supreme Court documents


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Current decoding: TEMPERATURE=0.3, TOP_P=0.9, REPETITION_PENALTY=1.0, NO_REPEAT_NGRAM=4
  BERTScore F1      : 0.8274
  ROUGE-L           : 0.2225
  BLEU-4            : 5.9203
  repeated 4-grams  : 0.0002
  (reference text itself: repeated 4-grams = 0.0142)


/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


### Decoding sweep -- choose the decoding settings from numbers

Greedy decoding always takes the single most likely token, and small models often get stuck in loops
(the same phrase again and again). The options compared below:

* **`NO_REPEAT_NGRAM`** forbids repeating any exact sequence of N tokens *within the continuation* --
  surgical: loops become impossible, ordinary words can still be reused.
* **`REPETITION_PENALTY`** makes every already-used token less likely, including "the" and "of",
  so it should stay small (about 1.05-1.15) or the text turns odd.
* **Temperature + top-p (nucleus) sampling** adds controlled randomness: the next token is drawn from the
  smallest set of tokens covering `TOP_P` of the probability mass. Low temperature (0.2-0.5) keeps text
  close to the model's best guess. Very fluent-looking sampled text can still score *lower* on
  reference-based metrics than a safe greedy continuation, which is exactly why this is measured.

Each configuration generates the same local examples with the same seed. **Read the table, decide which
trade-off you want (BERTScore / ROUGE / BLEU vs repetition), then set the knobs
(`TEMPERATURE` in Settings; `TOP_P`, `REPETITION_PENALTY`, `NO_REPEAT_NGRAM` in Section 13) and re-run
Section 13's "generate all" cell and the packaging cell.** The sweep needs only the trained model and
the generation functions, so it can also be run *before* the "generate all" cell. It takes a few minutes
per configuration on a T4; set `RUN_DECODING_SWEEP = False` to skip it.

In [24]:
RUN_DECODING_SWEEP = True
SWEEP_EXAMPLES = 100    # examples per configuration (first N of the local set); raise for less noisy numbers

# name, TEMPERATURE, TOP_P, REPETITION_PENALTY, NO_REPEAT_NGRAM
SWEEP_CONFIGS = [
    ("greedy",                       0.0, 1.00, 1.00, 0),
    ("greedy + no-repeat 4-gram",    0.0, 1.00, 1.00, 4),
    ("greedy + rep. penalty 1.05",   0.0, 1.00, 1.05, 0),
    ("greedy + rep. penalty 1.15",   0.0, 1.00, 1.15, 0),
    ("T 0.3, top-p 0.9",             0.3, 0.90, 1.00, 0),
    ("T 0.3, top-p 0.9 + 4-gram",    0.3, 0.90, 1.00, 4),
    ("T 0.5, top-p 0.9 + 4-gram",    0.5, 0.90, 1.00, 4),
    ("T 0.7, top-p 0.9 + 4-gram",    0.7, 0.90, 1.00, 4),
    ("starter: T 0.1, top-p 0.95",   0.1, 0.95, 1.00, 0),
]

sweep_results = {}
if not RUN_DECODING_SWEEP or not local_prompts:
    print("Decoding sweep skipped (RUN_DECODING_SWEEP is False, or there are no local examples).")
else:
    saved_knobs = (TEMPERATURE, TOP_P, REPETITION_PENALTY, NO_REPEAT_NGRAM)
    sweep_prompts, sweep_refs = local_prompts[:SWEEP_EXAMPLES], local_references[:SWEEP_EXAMPLES]
    try:
        for name, temp, top_p, rep_pen, ngram_n in SWEEP_CONFIGS:
            TEMPERATURE, TOP_P, REPETITION_PENALTY, NO_REPEAT_NGRAM = temp, top_p, rep_pen, ngram_n
            preds = generate_continuations_batch(sweep_prompts, num_words=128,
                                                  batch_size=GEN_BATCH_SIZE, verbose=False)
            sweep_results[name] = score_local(preds, sweep_refs)
            print(f"done: {name}")
    finally:
        TEMPERATURE, TOP_P, REPETITION_PENALTY, NO_REPEAT_NGRAM = saved_knobs   # leave the notebook's settings untouched

    columns = ["BERTScore F1", "ROUGE-L", "BLEU-4", "repeated 4-grams"]
    print()
    print(f"{'configuration':30s}" + "".join(f"{c:>18s}" for c in columns))
    for name, scores in sweep_results.items():
        print(f"{name:30s}" + "".join(f"{scores[c]:18.4f}" for c in columns))
    print(f"\n(reference text itself: repeated 4-grams = {repeated_4gram_rate(sweep_refs):.4f}; "
          f"{len(sweep_prompts)} examples per configuration)")
    print("Restored settings: "
          f"TEMPERATURE={TEMPERATURE}, TOP_P={TOP_P}, REPETITION_PENALTY={REPETITION_PENALTY}, NO_REPEAT_NGRAM={NO_REPEAT_NGRAM}")

if sweep_results:
    log_event("decoding_sweep", examples=SWEEP_EXAMPLES, results=sweep_results)
    sync_run_log()

done: greedy
done: greedy + no-repeat 4-gram
done: greedy + rep. penalty 1.05
done: greedy + rep. penalty 1.15
done: T 0.3, top-p 0.9
done: T 0.3, top-p 0.9 + 4-gram
done: T 0.5, top-p 0.9 + 4-gram
done: T 0.7, top-p 0.9 + 4-gram
done: starter: T 0.1, top-p 0.95

configuration                       BERTScore F1           ROUGE-L            BLEU-4  repeated 4-grams
greedy                                    0.7992            0.2126            4.0543            0.7530
greedy + no-repeat 4-gram                 0.8274            0.2292            6.2461            0.0000
greedy + rep. penalty 1.05                0.8076            0.2185            4.3895            0.6566
greedy + rep. penalty 1.15                0.8291            0.2194            6.8349            0.1858
T 0.3, top-p 0.9                          0.8012            0.2089            4.1488            0.6686
T 0.3, top-p 0.9 + 4-gram                 0.8274            0.2265            5.9950            0.0000
T 0.5, top-p 0.

/tmp/ipykernel_23/619487680.py:75: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## What to try next (this is where the real grade comes from)

The brief cares more about what you tried and learned than the raw score.
Once this notebook's real run (`QUICK_TEST = False`) works end to end, easy
things to change and re-run:

- **`NORM_TYPE` / `NORM_POSITION`** — try `rmsnorm` and `deepnorm`, and
  `pre` vs `post`. Compare training stability (does the loss curve look
  clean or does it spike?) and the local eval scores.
- **`ATTENTION_TYPE`** (this notebook, if it has one) — try `gqa` and
  `mqa` and see how much of the score they trade away for fewer
  key/value heads.
- **`OPTIMIZER_TYPE`** — compare `muon` against `adamw`: does it converge
  faster (look at the loss printed every 100 batches)?
- **`TEMPERATURE`** in Settings — try 0.7 or 1.0 instead of 0 (greedy) and
  compare the generated text and the local eval scores.
- **`VOCAB_SIZE`** — a bigger vocabulary means fewer `<unk>` words but a
  bigger embedding table.
- **`MAX_WORDS_FOR_TRAINING`** in the `else` branch of Settings — push this
  up as your GPU/time budget allows.
- **Which of the 4 sources you train on** — right now `TRAIN_FILES` walks
  through all 4 in a fixed order. Try filtering to just Supreme Court
  judgments, or just Acts, and see what changes.
- **Model size / `NUM_EPOCHS` / `LEARNING_RATE` / `BATCH_SIZE`** — the classic knobs.
- Once all 4 of your architecture notebooks have a real (`QUICK_TEST = False`)
  run with a local BERTScore/ROUGE-L/BLEU-4 number (Step 15), compare them
  and decide which model(s) you actually submit to Codabench and carry into
  Phase 2.

Remember: **minimum 4 real submissions** to Codabench before the Phase 1
deadline, and keep notes on what you tried and what happened (including
these new norm/attention/optimizer settings) in each notebook — that's
what goes into your final report.